In [1]:
library(Rcpp)
library(progress)
library(RcppEigen)
library(RcppDist)
library(RcppArmadillo)
library(mvtnorm)
library(dbarts)
sourceCpp("FirstModel.cpp")

Warning message:
"package 'Rcpp' was built under R version 4.3.3"
Warning message:
"package 'progress' was built under R version 4.3.3"
Warning message:
"package 'RcppEigen' was built under R version 4.3.3"
Warning message:
"package 'RcppDist' was built under R version 4.3.3"
Registered S3 methods overwritten by 'RcppArmadillo':
  method               from     
  predict.fastLm       RcppEigen
  print.fastLm         RcppEigen
  summary.fastLm       RcppEigen
  print.summary.fastLm RcppEigen


Attaching package: 'RcppArmadillo'


The following objects are masked from 'package:RcppEigen':

    fastLm, fastLmPure


Warning message:
"package 'mvtnorm' was built under R version 4.3.3"


# DGP_2

In [2]:
#Define Helper Functions
in_cred<-function(samples, value, interval)
{
  upper_quantile<-1-(1-interval)/2
  lower_quantile<-0+(1-interval)/2

  q1<-quantile(samples, lower_quantile)
  q2<-quantile(samples, upper_quantile)

  in_cred<-ifelse(value>=q1 & value<=q2, T, F)
}

cred_width<-function(samples, interval)
{
  upper_quantile<-1-(1-interval)/2
  lower_quantile<-0+(1-interval)/2

  q1<-quantile(samples, lower_quantile)
  q2<-quantile(samples, upper_quantile)

  return(q2-q1)
}

# Number of simulations
num_simulations <- 100

# Initialize a matrix to store the results
results_matrix <- matrix(NA, nrow = num_simulations, ncol = 30)
colnames(results_matrix) <- c("mvbcf_1k_pehe1", "mvbcf_1k_pehe2","mvbcf_0.5k_pehe1", "mvbcf_0.5k_pehe2","mvbcf_0.25k_pehe1", "mvbcf_0.25k_pehe2","mvbcf_0.1k_pehe1", "mvbcf_0.1k_pehe2",
                             "mvbcf_0.05k_pehe1", "mvbcf_0.05k_pehe2",
                             "mvbcf_1k_tau_951", "mvbcf_1k_tau_952","mvbcf_0.5k_tau_951", "mvbcf_0.5k_tau_952","mvbcf_0.25k_tau_951", "mvbcf_0.25k_tau_952","mvbcf_0.1k_tau_951", "mvbcf_0.1k_tau_952",
                             "mvbcf_0.05k_tau_951", "mvbcf_0.05k_tau_952", "mvbcf_1k_tau_951w", "mvbcf_1k_tau_952w","mvbcf_0.5k_tau_951w", "mvbcf_0.5k_tau_952w","mvbcf_0.25k_tau_951w", "mvbcf_0.25k_tau_952w","mvbcf_0.1k_tau_951w", "mvbcf_0.1k_tau_952w",
                             "mvbcf_0.05k_tau_951w", "mvbcf_0.05k_tau_952w")

# Create a progress bar
pb <- progress_bar$new(total = num_simulations)

# For loop to run the code 100 times
for (i in 1:num_simulations) {
  # Update progress bar
  pb$tick()
#Set random seed
seed_val<-i
set.seed(seed_val)

#Train Data
n<-500

X1<-runif(n)
X2<-runif(n)
X3<-runif(n)
X4<-runif(n)
X5<-runif(n)
X6<-rbinom(n, 1, 0.5)
X7<-rbinom(n, 1, 0.5)
X8<-rbinom(n, 1, 0.5)
X9<-sample(c(0, 1, 2, 3, 4), n, replace=T)
X10<-sample(c(0, 1, 2, 3, 4), n, replace=T)

X<-cbind(X1, X2, X3, X5, X6, X7, X8, X9, X10)

Mu1<-(11*sin(pi*X1*X2)+18*(X3-0.5)^2+10*X4+12*X6+X9)*10+300
Mu2<-(9*sin(pi*X1*X2)+22*(X3-0.5)^2+0*X4+8*X6+X9)*10+300

Tau1<-(2*X4+2*X5)*10
Tau2<-(1*X4+3*X5)*10

true_propensity<-X4

Z<-rbinom(n, 1, true_propensity)

Y<-cbind(Mu1+Z*Tau1, Mu2+Z*Tau2) + mvtnorm::rmvnorm(n, c(0, 0), matrix(c(50^2, 0, 0, 50^2), nrow=2, byrow=T))

#Test Data
n_test<-1000

X1_test<-runif(n_test)
X2_test<-runif(n_test)
X3_test<-runif(n_test)
X4_test<-runif(n_test)
X5_test<-runif(n_test)
X6_test<-rbinom(n_test, 1, 0.5)
X7_test<-rbinom(n_test, 1, 0.5)
X8_test<-rbinom(n_test, 1, 0.5)
X9_test<-sample(c(0, 1, 2, 3, 4), n_test, replace=T)
X10_test<-sample(c(0, 1, 2, 3, 4), n_test, replace=T)

X_test<-cbind(X1_test, X2_test, X3_test, X5_test, X6_test, X7_test, X8_test, X9_test, X10_test)

Mu1_test<-(11*sin(pi*X1_test*X2_test)+18*(X3_test-0.5)^2+10*X4_test+12*X6_test+X9_test)*10+300
Mu2_test<-(9*sin(pi*X1_test*X2_test)+22*(X3_test-0.5)^2+0*X4_test+8*X6_test+X9_test)*10+300

Tau1_test<-(2*X4_test+2*X5_test)*10
Tau2_test<-(1*X4_test+3*X5_test)*10

true_propensity_test<-X4_test

Z_test<-rbinom(n_test, 1, true_propensity_test)

Y_test<-cbind(Mu1_test+Z_test*Tau1_test, Mu2_test+Z_test*Tau2_test) + mvtnorm::rmvnorm(n_test, c(0, 0), matrix(c(50^2, 0, 0, 50^2), nrow=2, byrow=T))

#estimate of propensity score
p_mod<-bart(x.train = X, y.train = Z, x.test = X_test, k=3, verbose = FALSE)
p<-colMeans(pnorm(p_mod$yhat.train))
p_test<-colMeans(pnorm(p_mod$yhat.test))

#adding to matrix
X2<-X
X2_test<-X_test
X<-cbind(X, p)
X_test<-cbind(X_test, p_test)
Z2<-cbind(Z,Z)

#set some parameters
n_tree_mu<-50
n_tree_tau<-20
n_iter<-50
n_burn<-0
num_gfr<-1000

mu_val<-1
tau_val<-0.375
v_val<-1
wish_val<-1
min_val<-1

mvbcf_1k_mod <- fast_bart(X,
                         Y,
                         Z2,
                         X2,
                         X_test, # here is the test data for the mu part of the model
                         X2_test, # here is the test data for the tau part of the model
                         0.95,
                         2,
                         0.25,
                         3,
                         diag((mu_val)^2/n_tree_mu, 2),
                         diag((tau_val)^2/n_tree_tau, 2),
                         v_val,
                         diag(wish_val, 2),
                         n_iter,
                         n_tree_mu,
                         n_tree_tau,
                         min_val,
                         num_gfr)


mvbcf_1k_tau_preds1<-rowMeans(mvbcf_1k_mod$predictions_tau_test[,1,-c(1:n_burn)])
mvbcf_1k_ate1<-mean(mvbcf_1k_tau_preds1)
mvbcf_1k_tau_preds2<-rowMeans(mvbcf_1k_mod$predictions_tau_test[,2,-c(1:n_burn)])
mvbcf_1k_ate2<-mean(mvbcf_1k_tau_preds2)

mvbcf_1k_pehe1<-sqrt(mean((Tau1_test-mvbcf_1k_tau_preds1)^2))
mvbcf_1k_pehe2<-sqrt(mean((Tau2_test-mvbcf_1k_tau_preds2)^2))

ate1 <- mean(Tau1_test)
ate2 <- mean(Tau2_test)

mvbcf_1k_tau_951<-mean(diag(apply(mvbcf_1k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, in_cred, Tau1_test, 0.95)))
mvbcf_1k_tau_951w<-mean(apply(mvbcf_1k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, cred_width, 0.95))

mvbcf_1k_tau_952<-mean(diag(apply(mvbcf_1k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, in_cred, Tau2_test, 0.95)))
mvbcf_1k_tau_952w<-mean(apply(mvbcf_1k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, cred_width, 0.95))


n_iter<-50
n_burn<-0
num_gfr<-500

mvbcf_0.5k_mod <- fast_bart(X,
                         Y,
                         Z2,
                         X2,
                         X_test, # here is the test data for the mu part of the model
                         X2_test, # here is the test data for the tau part of the model
                         0.95,
                         2,
                         0.25,
                         3,
                         diag((mu_val)^2/n_tree_mu, 2),
                         diag((tau_val)^2/n_tree_tau, 2),
                         v_val,
                         diag(wish_val, 2),
                         n_iter,
                         n_tree_mu,
                         n_tree_tau,
                         min_val,
                         num_gfr)


mvbcf_0.5k_tau_preds1<-rowMeans(mvbcf_0.5k_mod$predictions_tau_test[,1,-c(1:n_burn)])
mvbcf_0.5k_ate1<-mean(mvbcf_0.5k_tau_preds1)
mvbcf_0.5k_tau_preds2<-rowMeans(mvbcf_0.5k_mod$predictions_tau_test[,2,-c(1:n_burn)])
mvbcf_0.5k_ate2<-mean(mvbcf_0.5k_tau_preds2)

mvbcf_0.5k_pehe1<-sqrt(mean((Tau1_test-mvbcf_0.5k_tau_preds1)^2))
mvbcf_0.5k_pehe2<-sqrt(mean((Tau2_test-mvbcf_0.5k_tau_preds2)^2))

ate1 <- mean(Tau1_test)
ate2 <- mean(Tau2_test)

mvbcf_0.5k_tau_951<-mean(diag(apply(mvbcf_0.5k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, in_cred, Tau1_test, 0.95)))
mvbcf_0.5k_tau_951w<-mean(apply(mvbcf_0.5k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, cred_width, 0.95))

mvbcf_0.5k_tau_952<-mean(diag(apply(mvbcf_0.5k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, in_cred, Tau2_test, 0.95)))
mvbcf_0.5k_tau_952w<-mean(apply(mvbcf_0.5k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, cred_width, 0.95))


n_iter<-50
n_burn<-0
num_gfr<-250

mvbcf_0.25k_mod <- fast_bart(X,
                         Y,
                         Z2,
                         X2,
                         X_test, # here is the test data for the mu part of the model
                         X2_test, # here is the test data for the tau part of the model
                         0.95,
                         2,
                         0.25,
                         3,
                         diag((mu_val)^2/n_tree_mu, 2),
                         diag((tau_val)^2/n_tree_tau, 2),
                         v_val,
                         diag(wish_val, 2),
                         n_iter,
                         n_tree_mu,
                         n_tree_tau,
                         min_val,
                         num_gfr)


mvbcf_0.25k_tau_preds1<-rowMeans(mvbcf_0.25k_mod$predictions_tau_test[,1,-c(1:n_burn)])
mvbcf_0.25k_ate1<-mean(mvbcf_0.25k_tau_preds1)
mvbcf_0.25k_tau_preds2<-rowMeans(mvbcf_0.25k_mod$predictions_tau_test[,2,-c(1:n_burn)])
mvbcf_0.25k_ate2<-mean(mvbcf_0.25k_tau_preds2)

mvbcf_0.25k_pehe1<-sqrt(mean((Tau1_test-mvbcf_0.25k_tau_preds1)^2))
mvbcf_0.25k_pehe2<-sqrt(mean((Tau2_test-mvbcf_0.25k_tau_preds2)^2))

ate1 <- mean(Tau1_test)
ate2 <- mean(Tau2_test)

mvbcf_0.25k_tau_951<-mean(diag(apply(mvbcf_0.25k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, in_cred, Tau1_test, 0.95)))
mvbcf_0.25k_tau_951w<-mean(apply(mvbcf_0.25k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, cred_width, 0.95))

mvbcf_0.25k_tau_952<-mean(diag(apply(mvbcf_0.25k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, in_cred, Tau2_test, 0.95)))
mvbcf_0.25k_tau_952w<-mean(apply(mvbcf_0.25k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, cred_width, 0.95))


n_iter<-50
n_burn<-0
num_gfr<-100

mvbcf_0.1k_mod <- fast_bart(X,
                         Y,
                         Z2,
                         X2,
                         X_test, # here is the test data for the mu part of the model
                         X2_test, # here is the test data for the tau part of the model
                         0.95,
                         2,
                         0.25,
                         3,
                         diag((mu_val)^2/n_tree_mu, 2),
                         diag((tau_val)^2/n_tree_tau, 2),
                         v_val,
                         diag(wish_val, 2),
                         n_iter,
                         n_tree_mu,
                         n_tree_tau,
                         min_val,
                         num_gfr)


mvbcf_0.1k_tau_preds1<-rowMeans(mvbcf_0.1k_mod$predictions_tau_test[,1,-c(1:n_burn)])
mvbcf_0.1k_ate1<-mean(mvbcf_0.1k_tau_preds1)
mvbcf_0.1k_tau_preds2<-rowMeans(mvbcf_0.1k_mod$predictions_tau_test[,2,-c(1:n_burn)])
mvbcf_0.1k_ate2<-mean(mvbcf_0.1k_tau_preds2)

mvbcf_0.1k_pehe1<-sqrt(mean((Tau1_test-mvbcf_0.1k_tau_preds1)^2))
mvbcf_0.1k_pehe2<-sqrt(mean((Tau2_test-mvbcf_0.1k_tau_preds2)^2))

ate1 <- mean(Tau1_test)
ate2 <- mean(Tau2_test)

mvbcf_0.1k_tau_951<-mean(diag(apply(mvbcf_0.1k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, in_cred, Tau1_test, 0.95)))
mvbcf_0.1k_tau_951w<-mean(apply(mvbcf_0.1k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, cred_width, 0.95))

mvbcf_0.1k_tau_952<-mean(diag(apply(mvbcf_0.1k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, in_cred, Tau2_test, 0.95)))
mvbcf_0.1k_tau_952w<-mean(apply(mvbcf_0.1k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, cred_width, 0.95))


n_iter<-50
n_burn<-0
num_gfr<-50

mvbcf_0.05k_mod <- fast_bart(X,
                         Y,
                         Z2,
                         X2,
                         X_test, # here is the test data for the mu part of the model
                         X2_test, # here is the test data for the tau part of the model
                         0.95,
                         2,
                         0.25,
                         3,
                         diag((mu_val)^2/n_tree_mu, 2),
                         diag((tau_val)^2/n_tree_tau, 2),
                         v_val,
                         diag(wish_val, 2),
                         n_iter,
                         n_tree_mu,
                         n_tree_tau,
                         min_val,
                         num_gfr)


mvbcf_0.05k_tau_preds1<-rowMeans(mvbcf_0.05k_mod$predictions_tau_test[,1,-c(1:n_burn)])
mvbcf_0.05k_ate1<-mean(mvbcf_0.05k_tau_preds1)
mvbcf_0.05k_tau_preds2<-rowMeans(mvbcf_0.05k_mod$predictions_tau_test[,2,-c(1:n_burn)])
mvbcf_0.05k_ate2<-mean(mvbcf_0.05k_tau_preds2)

mvbcf_0.05k_pehe1<-sqrt(mean((Tau1_test-mvbcf_0.05k_tau_preds1)^2))
mvbcf_0.05k_pehe2<-sqrt(mean((Tau2_test-mvbcf_0.05k_tau_preds2)^2))

ate1 <- mean(Tau1_test)
ate2 <- mean(Tau2_test)

mvbcf_0.05k_tau_951<-mean(diag(apply(mvbcf_0.05k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, in_cred, Tau1_test, 0.95)))
mvbcf_0.05k_tau_951w<-mean(apply(mvbcf_0.05k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, cred_width, 0.95))

mvbcf_0.05k_tau_952<-mean(diag(apply(mvbcf_0.05k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, in_cred, Tau2_test, 0.95)))
mvbcf_0.05k_tau_952w<-mean(apply(mvbcf_0.05k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, cred_width, 0.95))


# Store the results in the matrix
  results_matrix[i, ] <- c(mvbcf_1k_pehe1, mvbcf_1k_pehe2, mvbcf_0.5k_pehe1, mvbcf_0.5k_pehe2, mvbcf_0.25k_pehe1, mvbcf_0.25k_pehe2, mvbcf_0.1k_pehe1, mvbcf_0.1k_pehe2,
                             mvbcf_0.05k_pehe1, mvbcf_0.05k_pehe2,
                             mvbcf_1k_tau_951, mvbcf_1k_tau_952, mvbcf_0.5k_tau_951, mvbcf_0.5k_tau_952, mvbcf_0.25k_tau_951, mvbcf_0.25k_tau_952, mvbcf_0.1k_tau_951, mvbcf_0.1k_tau_952,
                             mvbcf_0.05k_tau_951, mvbcf_0.05k_tau_952, mvbcf_1k_tau_951w, mvbcf_1k_tau_952w, mvbcf_0.5k_tau_951w, mvbcf_0.5k_tau_952w, mvbcf_0.25k_tau_951w, mvbcf_0.25k_tau_952w, mvbcf_0.1k_tau_951w, mvbcf_0.1k_tau_952w,
                             mvbcf_0.05k_tau_951w, mvbcf_0.05k_tau_952w)

}

# Export the results matrix to a CSV file
write.csv(results_matrix, "XMVBCF_simulation_results_DGP2.csv", row.names = FALSE)

# Print a message indicating completion
cat("Simulation completed and results saved to XMVBCF_simulation_results_DGP2.csv\n")

Warning message in validateXTest(test, attr(x, "term.labels"), ncol(x), colnames(x), :
"column names of 'test' does not equal that of 'x': 'X1, X2, X3, X5, X6, X7, X8, X9, X10'; match will be made by position"


Running GFR warm-start (1000 iterations)...
GFR warm-start done!

Total MVBCF runtime: 53123 ms
Running GFR warm-start (500 iterations)...
GFR warm-start done!

Total MVBCF runtime: 31388 ms
Running GFR warm-start (250 iterations)...
GFR warm-start done!

Total MVBCF runtime: 15161 ms
Running GFR warm-start (100 iterations)...
GFR warm-start done!

Total MVBCF runtime: 6347 ms
Running GFR warm-start (50 iterations)...
GFR warm-start done!

Total MVBCF runtime: 4045 ms


Warning message in validateXTest(test, attr(x, "term.labels"), ncol(x), colnames(x), :
"column names of 'test' does not equal that of 'x': 'X1, X2, X3, X5, X6, X7, X8, X9, X10'; match will be made by position"


Running GFR warm-start (1000 iterations)...
GFR warm-start done!

Total MVBCF runtime: 47807 ms
Running GFR warm-start (500 iterations)...
GFR warm-start done!

Total MVBCF runtime: 25185 ms
Running GFR warm-start (250 iterations)...
GFR warm-start done!

Total MVBCF runtime: 13525 ms
Running GFR warm-start (100 iterations)...
GFR warm-start done!

Total MVBCF runtime: 6344 ms
Running GFR warm-start (50 iterations)...
GFR warm-start done!

Total MVBCF runtime: 4054 ms


Warning message in validateXTest(test, attr(x, "term.labels"), ncol(x), colnames(x), :
"column names of 'test' does not equal that of 'x': 'X1, X2, X3, X5, X6, X7, X8, X9, X10'; match will be made by position"


...

Running GFR warm-start (1000 iterations)...
GFR warm-start done!

Total MVBCF runtime: 49730 ms
Running GFR warm-start (500 iterations)...
GFR warm-start done!

Total MVBCF runtime: 26525 ms
Running GFR warm-start (250 iterations)...
GFR warm-start done!

Total MVBCF runtime: 14322 ms
Running GFR warm-start (100 iterations)...
GFR warm-start done!

Total MVBCF runtime: 6714 ms
Running GFR warm-start (50 iterations)...
GFR warm-start done!

Total MVBCF runtime: 4213 ms


Warning message in validateXTest(test, attr(x, "term.labels"), ncol(x), colnames(x), :
"column names of 'test' does not equal that of 'x': 'X1, X2, X3, X5, X6, X7, X8, X9, X10'; match will be made by position"


Running GFR warm-start (1000 iterations)...
GFR warm-start done!

Total MVBCF runtime: 50943 ms
Running GFR warm-start (500 iterations)...
GFR warm-start done!

Total MVBCF runtime: 26418 ms
Running GFR warm-start (250 iterations)...
GFR warm-start done!

Total MVBCF runtime: 14519 ms
Running GFR warm-start (100 iterations)...
GFR warm-start done!

Total MVBCF runtime: 6776 ms
Running GFR warm-start (50 iterations)...
GFR warm-start done!

Total MVBCF runtime: 4148 ms


Warning message in validateXTest(test, attr(x, "term.labels"), ncol(x), colnames(x), :
"column names of 'test' does not equal that of 'x': 'X1, X2, X3, X5, X6, X7, X8, X9, X10'; match will be made by position"


Running GFR warm-start (1000 iterations)...
GFR warm-start done!

Total MVBCF runtime: 50428 ms
Running GFR warm-start (500 iterations)...
GFR warm-start done!

Total MVBCF runtime: 26154 ms
Running GFR warm-start (250 iterations)...
GFR warm-start done!

Total MVBCF runtime: 13806 ms
Running GFR warm-start (100 iterations)...
GFR warm-start done!

Total MVBCF runtime: 6707 ms
Running GFR warm-start (50 iterations)...
GFR warm-start done!

Total MVBCF runtime: 4055 ms
Simulation completed and results saved to XMVBCF_simulation_results_DGP2.csv


1m and 7.5s per iteration

1 hour and 53 minutes per 100 replications

In [3]:
print(results_matrix)

       mvbcf_1k_pehe1 mvbcf_1k_pehe2 mvbcf_0.5k_pehe1 mvbcf_0.5k_pehe2
  [1,]       38.73213      10.061317         37.70094         9.232926
  [2,]       44.92148       7.436694         45.08624         6.879378
  [3,]       36.53107       7.746830         36.44356         8.570593
  [4,]       46.61504       9.344214         47.20659         9.723108
  [5,]       34.62420       8.788350         37.80821         9.016164
  [6,]       47.32734       8.182594         46.29284         7.447544
  [7,]       39.08847       7.666116         39.94184         8.985736
  [8,]       36.06539       9.679306         37.45585         8.389175
  [9,]       38.28670       7.782948         39.38772         7.705928
 [10,]       33.55037      10.532753         31.96102        10.985925
 [11,]       37.07176      11.574192         32.89671        11.755333
 [12,]       34.35326       9.238633         34.59558        11.447419
 [13,]       29.97388      11.966488         32.02208        10.727474
 [14,]